# 07_attention_and_transformer_prerequisites: Scaled Dot-Product Attention Implementation

This notebook validates scaled dot-product attention mechanics. It implements the attention alignment projection loop in PyTorch, verifying it against our step-by-step hand calculations.

## 1. Scaled Dot-Product Attention in PyTorch

In [1]:
import torch
import torch.nn.functional as F
import numpy as np

# Define Query, Keys, and Values matching our hand-calculations exactly
q = torch.tensor([[1.0, 2.0]]) # shape (1, d_k)
k = torch.tensor([[1.0, 0.0],  # k1
                  [0.0, 2.0]]) # k2, shape (L, d_k)
v = torch.tensor([[10.0, 20.0],
                  [30.0, 40.0]]) # shape (L, d_v)

d_k = q.size(-1)

# 1. Compute scores (q * K^T)
scores = torch.matmul(q, k.t())

# 2. Scale by sqrt(d_k)
scaled_scores = scores / (d_k ** 0.5)

# 3. Softmax attention weights
weights = F.softmax(scaled_scores, dim=-1)

# 4. Weighted sum of values
context = torch.matmul(weights, v)

print("Raw scores:        ", scores.numpy().flatten())
print("Scaled scores:     ", scaled_scores.numpy().flatten())
print("Attention weights: ", weights.numpy().flatten())
print("Context vector:    ", context.numpy().flatten())

# Assertions validating consistency with hand-calculations (context matches [27.8592, 37.8592])
np.testing.assert_almost_equal(context.numpy()[0], [27.8592, 37.8592], decimal=4)

Raw scores:         [1. 4.]
Scaled scores:      [0.70710677 2.828427  ]
Attention weights:  [0.10704181 0.89295816]
Context vector:     [27.859163 37.85916 ]


### Output Analysis: Attention Retrieval
The PyTorch execution matches our hand calculations. The query $\mathbf{q} = [1, 2]^T$ has a much stronger dot-product alignment with $\mathbf{k}_2$ than $\mathbf{k}_1$, leading to scaled scores of $0.7071$ and $2.8284$. The resulting softmax attention weights assign $89.30\%$ of the probability to index 2, projecting the final output context vector to `[27.8592, 37.8592]` (equal to `[27.8600, 37.8600]` if intermediate values are rounded), verifying execution consistency.